# 1D Excitatory-Inhibitory Network

This notebook extends the linear model to an **excitatory-inhibitory (E–I) network**.

## Idea

We model interactions between:
- **Excitatory neurons (E)** → amplify activity  
- **Inhibitory neurons (I)** → suppress activity  

The balance between them determines whether the network shows **amplification or suppression**.

## Model

The recurrent connectivity includes:
- E → E, E → I  
- I → E, I → I  

In [ ]:
import numpy as np
from scipy.linalg import circulant
import matplotlib.pyplot as plt
from numpy.fft import fft, ifft
import seaborn as sns
from scipy.spatial.distance import cdist
from scipy import signal
from random import gauss
from pandas import Series
from src.ExcitatoryInhibitoryModel1D import gaussian, recurrent_connections, convert_matrix, not_rescaled, compute_firerate
from src.ExcitatoryInhibitoryModel1D import F, perturb_C, make_gain_vector, perturb_C_pointwise
from src.ExcitatoryInhibitoryModel1D import compute_variance_connectivity_recentered, compute_variance_influence_recentered

In [ ]:
# Configuration
N = 200
alpha = 0.97
I = np.eye(2 * N)

# strength parameter such that: (1 + W_II)W_EE < W_EIW_IE

W_EE = recurrent_connections(N, a = 1.0, sigma = 2.5)
W_EI = recurrent_connections(N, a = 2.0, sigma = 4.0)
W_IE = recurrent_connections(N, a = 2.0, sigma = 1.5 * 2.5)
W_II = recurrent_connections(N, a = 0.5, sigma = 1.7 * 4.0)

C = convert_matrix(W_EE, W_EI, W_IE, W_II, alpha)

# Perturbation
corr = F(N)          
C_het = perturb_C_pointwise(C, N, seed=42, amplitude=0.2)

In [ ]:
# INPUT
x = np.arange(N)

A_e, A_i = 1.0, 0.8      # Amplituden
k_e, k_i = 4, 8          # Anzahl Perioden auf dem Ring (räumliche Frequenzen)

h_e = A_e * np.cos(2 * np.pi * k_e * x / N)
h_i = A_i * np.cos(2 * np.pi * k_i * x / N)

h = np.concatenate((h_e, h_i))

r = np.linalg.solve(I - C, h)
r_e = r[:N]
r_i = r[N:]

In [ ]:
# Plot Connectivity Kernels

half = N // 2
x = np.arange(-half, half)  

matrices = [W_EE, -W_EI, W_IE, -W_II]
titles   = ['$W_{EE}$', '$W_{EI}$', '$W_{IE}$', '$W_{II}$']
colors   = ['steelblue', '#C0392B', '#27AE60', '#E67E22']

fig, axes = plt.subplots(2, 1, figsize=(10, 7))

ax = axes[0]
for mat, title, color in zip(matrices, titles, colors):
    row = np.roll(mat[0], half)          
    ax.plot(x, row, color=color, linewidth=2, label=title)
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_title('Non-perturbed Connectivity Kernels', fontsize=13, fontweight='bold')
ax.text(-0.06, 1.02, '(A)', transform=ax.transAxes, fontsize=14, fontweight='bold')
ax.set_ylabel('Synaptic weight', fontsize=12)
ax.set_xlim(0, 20)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
for spine in ax.spines.values():
    spine.set_linewidth(2)

ax = axes[1]
W_EE_p = C_het[:N,   :N]
W_EI_p = C_het[:N,   N:2*N]
W_IE_p = C_het[N:2*N, :N]
W_II_p = C_het[N:2*N, N:2*N]

matrices_p = [W_EE_p, W_EI_p, W_IE_p, W_II_p]

for mat, title, color in zip(matrices_p, titles, colors):
    row = np.roll(mat[0], half)          
    ax.plot(x, row, color=color, linewidth=2, label=title)
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_title('Perturbed Connectivity Kernels', fontsize=13, fontweight='bold')
ax.text(-0.06, 1.02, '(B)', transform=ax.transAxes, fontsize=14, fontweight='bold')
ax.set_xlabel('Distance from neuron $i$', fontsize=12)
ax.set_ylabel('Synaptic weight', fontsize=12)
ax.set_xlim(0, 20)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
for spine in ax.spines.values():
    spine.set_linewidth(2)

plt.suptitle('Connectivity Kernels — E-I Network', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("EI_kernels.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
#### Non-Perturbed Kernels
half = N // 2
x    = np.arange(-half, half)

matrices  = [W_EE, -W_EI, W_IE, -W_II]
titles    = ['$W_{EE}$', '$W_{EI}$', '$W_{IE}$', '$W_{II}$']
labels    = ['(A)', '(B)', '(C)', '(D)']
colors    = ['steelblue', '#C0392B', '#27AE60', '#E67E22']

fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True)

for ax, mat, title, label, color in zip(axes.flat, matrices, titles, labels, colors):
    row = np.roll(mat[0], half)
    ax.plot(x, row, color=color, linewidth=2)
    ax.axhline(0, color='gray', linewidth=0.8)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.text(-0.08, 1.02, label, transform=ax.transAxes, fontsize=14, fontweight='bold')
    ax.set_xlim(0, half)
    ax.grid(True, alpha=0.3)
    for spine in ax.spines.values():
        spine.set_linewidth(2)

for ax in axes[1, :]:
    ax.set_xlabel('Distance from neuron $i$', fontsize=12)
for ax in axes[:, 0]:
    ax.set_ylabel('Synaptic weight', fontsize=12)

plt.suptitle('Connectivity Kernels — E-I Network', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("EI_kernels.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
### Influence

h_e = np.zeros(N, dtype=float)
h_i = np.zeros(N, dtype=float)
h_e[0] = 1.0
h = np.concatenate((h_e, h_i))

# Non-perturbed
r_hom   = np.linalg.solve(I - C, h)
# Perturbed
r_het   = np.linalg.solve(I - C_het, h)

def split_influence(r, h_e, h_i, N):
    r_e = r[:N];  r_i = r[N:]
    inf_e = r_e - h_e
    inf_i = r_i - h_i
    return r_e, r_i, inf_e, inf_i

r_e_hom, r_i_hom, inf_e_hom, inf_i_hom = split_influence(r_hom, h_e, h_i, N)
r_e_het, r_i_het, inf_e_het, inf_i_het = split_influence(r_het, h_e, h_i, N)

x = np.arange(20)

fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

ax = axes[0]
ax.plot(x, r_e_hom[:20],   color='tab:blue',   linewidth=2,      label='$r_E$ (response)')
ax.plot(x, inf_e_hom[:20], color='tab:cyan',   linewidth=2, ls='--', label='Influence $r_E$')
ax.plot(x, r_i_hom[:20],   color='tab:orange', linewidth=2,      label='$r_I$ (response)')
ax.plot(x, inf_i_hom[:20], color='tab:red',    linewidth=2, ls='--', label='Influence $r_I$')
ax.plot(x, h_e[:20],       color='black',      linewidth=1, ls=':',  label='Input')
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_ylabel('Influence', fontsize=12)
ax.set_title('Non-perturbed', fontsize=13, fontweight='bold')
ax.text(-0.06, 1.02, '(A)', transform=ax.transAxes, fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 7.5)
for spine in ax.spines.values():
    spine.set_linewidth(2)

ax = axes[1]
ax.plot(x, r_e_het[:20],   color='tab:blue',   linewidth=2,      label='$r_E$ (response)')
ax.plot(x, inf_e_het[:20], color='tab:cyan',   linewidth=2, ls='--', label='Influence $r_E$')
ax.plot(x, r_i_het[:20],   color='tab:orange', linewidth=2,      label='$r_I$ (response)')
ax.plot(x, inf_i_het[:20], color='tab:red',    linewidth=2, ls='--', label='Influence $r_I$')
ax.plot(x, h_e[:20],       color='black',      linewidth=1, ls=':',  label='Input')
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_xlabel('Position $x$', fontsize=12)
ax.set_ylabel('Influence', fontsize=12)
ax.set_title('Perturbed', fontsize=13, fontweight='bold')
ax.text(-0.06, 1.02, '(B)', transform=ax.transAxes, fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 7.5)
for spine in ax.spines.values():
    spine.set_linewidth(2)

plt.suptitle('Response to single-neuron stimulation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("influence.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
# Perturbation and Variance

G_hom = compute_influence_matrix_from_C(C)
G_het = compute_influence_matrix_from_C(C_het)

var_conn_E_hom, var_conn_I_hom = compute_variance_connectivity_recentered(C,     N)
var_conn_E_het, var_conn_I_het = compute_variance_connectivity_recentered(C_het, N)

var_inf_E_hom, var_inf_I_hom = compute_variance_influence_recentered(G_hom, N)
var_inf_E_het, var_inf_I_het = compute_variance_influence_recentered(G_het, N)

x = np.arange(N) - N//2   

fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

ax = axes[0]
ax.plot(x, var_conn_E_hom, color='tab:blue',   linewidth=2,
        label='$W_{EE}$ non-perturbed')
ax.plot(x, var_conn_I_hom, color='tab:orange', linewidth=2,
        label='$W_{II}$ non-perturbed')
ax.plot(x, var_conn_E_het, color='tab:blue',   linewidth=2, ls='--',
        label='$W_{EE}$ perturbed')
ax.plot(x, var_conn_I_het, color='tab:orange', linewidth=2, ls='--',
        label='$W_{II}$ perturbed')
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_ylabel('Variance', fontsize=12)
ax.set_title('Connectivity Variance', fontsize=13, fontweight='bold')
ax.text(-0.06, 1.02, '(A)', transform=ax.transAxes, fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 60)
for spine in ax.spines.values():
    spine.set_linewidth(2)

ax = axes[1]
ax.plot(x, var_inf_E_hom, color='tab:blue',   linewidth=2,
        label='$r_E$ non-perturbed')
ax.plot(x, var_inf_I_hom, color='tab:orange', linewidth=2,
        label='$r_I$ non-perturbed')
ax.plot(x, var_inf_E_het, color='tab:blue',   linewidth=2, ls='--',
        label='$r_E$ perturbed')
ax.plot(x, var_inf_I_het, color='tab:orange', linewidth=2, ls='--',
        label='$r_I$ perturbed')
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_xlabel('Neuron position $x$', fontsize=12)
ax.set_ylabel('Variance', fontsize=12)
ax.set_title('Influence Variance', fontsize=13, fontweight='bold')
ax.text(-0.06, 1.02, '(B)', transform=ax.transAxes, fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 60)
for spine in ax.spines.values():
    spine.set_linewidth(2)

plt.suptitle('Connectivity and Influence Variance — E-I Network',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("variance_EI.png", dpi=180, bbox_inches="tight")
plt.show()

print(f"Connectivity variance (non-perturbed) — E: {np.mean(var_conn_E_hom):.2e},  I: {np.mean(var_conn_I_hom):.2e}")
print(f"Connectivity variance (perturbed)     — E: {np.mean(var_conn_E_het):.2e},  I: {np.mean(var_conn_I_het):.2e}")
print(f"Influence variance    (non-perturbed) — E: {np.mean(var_inf_E_hom):.2e},  I: {np.mean(var_inf_I_hom):.2e}")
print(f"Influence variance    (perturbed)     — E: {np.mean(var_inf_E_het):.2e},  I: {np.mean(var_inf_I_het):.2e}")